<a href="https://colab.research.google.com/github/Okahak71/learning_ai/blob/main/makemore_MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch as t
import torch.nn.functional as F
import time

t.set_default_device('cuda')

print_every = 5000
start_time = time.time()
last_time = time.time()

words = open("names.txt", 'r').read().splitlines()
st = sorted(list(set(''.join(words))))
stoi = {s:i + 1 for i, s in enumerate(st)}
stoi["."] = 0
itos = {i : s for s, i in stoi.items()}

#Hyperparameters
learning_rate = 0.1
vocab_size = 27
block_size = 5
neurons = 200
dim = 10

#-------------
def build_dataset(words):
    xs, ys = [], []
    for w in words:
        context = [0] * block_size
        for c in w + '.':
            xs.append(context)
            ys.append(stoi[c])
            context = context[1:] + [stoi[c]]
    xs = t.tensor(xs)
    ys = t.tensor(ys)
    return xs, ys

#Training - 80%, Val/Dev - 10%, Testing - 10%
import random
random.seed(42)
random.shuffle(words)
sz1 = int(0.8 * len(words))
sz2 = int(0.9 * len(words))
x_tr, y_tr = build_dataset(words[:sz1])
x_dev, y_dev = build_dataset(words[sz1:sz2])
x_test, y_test = build_dataset(words[sz2:])

#-----------Parameters---------
rng = t.Generator(device='cuda').manual_seed(2147483647)
C = t.randn((vocab_size, dim), generator=rng) # THE Lookup Table

#first layer
w1 = t.randn((block_size * dim, neurons), generator=rng)
b1 = t.randn((neurons), generator=rng)

#second layer
w2 = t.randn((neurons, vocab_size), generator=rng)
b2 = t.randn(vocab_size, generator=rng)

parameters = [C, w1, b1, w2, b2]

for p in parameters:
    p.requires_grad = True
total = 0

In [4]:
for i in range(300000):
    total += 1
    # Minibatch construction
    id = t.randint(0, x_tr.shape[0], (128,))

    #forward pass
    emb = C[x_tr[id]]
    h = t.tanh(emb.view(-1, block_size * dim) @ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, y_tr[id])
    # print(loss.item())

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    lr = learning_rate / pow(10, (total / 70000))
    for p in parameters:
        p.data += -lr * p.grad

    #------------------------------
    if i > 0 and i % print_every == 0:
        current_time = time.time()
        elapsed_chunk = current_time - last_time
        iterations_per_second = print_every / elapsed_chunk
        print(f"\rIteration: {i} | Speed: {iterations_per_second:.2f} it/s | loss = {loss}")
        last_time = current_time

Iteration: 5000 | Speed: 360.55 it/s | loss = 2.564199447631836
Iteration: 10000 | Speed: 736.53 it/s | loss = 2.2248878479003906
Iteration: 15000 | Speed: 754.34 it/s | loss = 2.268946647644043
Iteration: 20000 | Speed: 713.13 it/s | loss = 2.2412021160125732
Iteration: 25000 | Speed: 769.67 it/s | loss = 2.4441401958465576
Iteration: 30000 | Speed: 717.02 it/s | loss = 2.180983066558838
Iteration: 35000 | Speed: 742.71 it/s | loss = 2.3430843353271484
Iteration: 40000 | Speed: 684.19 it/s | loss = 2.0936357975006104
Iteration: 45000 | Speed: 738.15 it/s | loss = 2.0580170154571533
Iteration: 50000 | Speed: 750.04 it/s | loss = 2.112699031829834
Iteration: 55000 | Speed: 775.57 it/s | loss = 2.2000536918640137
Iteration: 60000 | Speed: 705.90 it/s | loss = 2.1534109115600586
Iteration: 65000 | Speed: 768.69 it/s | loss = 2.1019206047058105
Iteration: 70000 | Speed: 746.53 it/s | loss = 2.2618069648742676
Iteration: 75000 | Speed: 747.91 it/s | loss = 2.0459933280944824
Iteration: 8000

In [5]:
emb = C[x_dev]
h = t.tanh(emb.view(-1, dim * block_size) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, y_dev)
print(loss.item())
print(f"current learning_rate : {lr}")
print(total)

2.2330851554870605
current learning_rate : 5.179474679231213e-06
300000
